In [ ]:
# ============================================================
# RQ2: Out-of-sample predictive comparison
# Linear Regression vs XGBoost
# ============================================================
#
# Research question:
# Under rigorous out-of-sample validation, how does a gradient
# boosted ensemble (XGBoost) compare with a linear baseline
# (Linear Regression) in predicting megaproject cost overruns,
# and what does the comparison imply for model selection under
# small, right-skewed samples?
#
# Validation:
#   - 10-fold cross-validation
#   - repeated 20 times with different random seeds
#   - each project receives one out-of-fold prediction per repeat
#
# Primary comparison:
#   - project-level mean absolute error (MAE)
#   - the project is the unit of paired inference
#   - paired bootstrap confidence interval
#   - paired permutation test
#   - Wilcoxon signed-rank test reported as a robustness check
#
# Target:
#   y* = ln(1 + Cost_Overrun_Pct / 100)
#
# Predictions are back-transformed to percentage points before
# calculating predictive performance.
#
# The on-budget model is retained as a descriptive benchmark,
# but it is NOT included in the LR-vs-XGBoost inferential test.
#
# Country is excluded because the current specification treats
# Source as the relevant categorical variable. This should be
# justified explicitly in the dissertation.
# ============================================================

import os
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold
from sklearn.base import clone

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)

from xgboost import XGBRegressor

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

In [ ]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

INPUT_XLSX = "Master_Dataset_Rebased_2025_v6.xlsx"
OUTPUT_XLSX = "RQ2_Model_Results_Corrected.xlsx"

FIG_DIR = "RQ2_Figures_Corrected"

SHEET = "Model_Ready"

# Repeated cross-validation
N_SPLITS = 10
N_REPEATS = 20

RANDOM_STATE = 42

# Practical prediction tolerances
TOLERANCES = [10, 25]

# Expected analytical sample
EXPECTED_N = 107

# Number of resamples for inferential sensitivity analysis
N_BOOTSTRAP = 20000
N_PERMUTATIONS = 20000

# Optional retransformation sensitivity analysis
#
# Keep False for the PRIMARY analysis to preserve comparability
# with the original specification.
#
# Set to True for a separate sensitivity analysis using
# fold-specific Duan smearing.
APPLY_DUAN_SMEARING = False

os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ------------------------------------------------------------
# 2. Load and filter data
# ------------------------------------------------------------

df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET
)

# Restrict to the defined megaproject sample
df = (
    df[df["Meets_500M_2025"] == "Yes"]
    .copy()
    .reset_index(drop=True)
)

if len(df) != EXPECTED_N:
    raise ValueError(
        f"Expected {EXPECTED_N} projects after filtering, "
        f"but found {len(df)}."
    )

print(f"Analytical sample: N = {len(df)} projects")

# Required variables
REQUIRED_COLUMNS = [
    "Cost_Overrun_Pct",
    "Orig_USD_PPP_2025_M",
    "Sector_Std",
    "Source",
]

missing = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# Check missingness in modelling variables
missing_summary = df[REQUIRED_COLUMNS].isna().sum()

print("\nMissing values:")
print(missing_summary)

Analytical sample: N = 107 projects

Missing values:
Cost_Overrun_Pct       0
Orig_USD_PPP_2025_M    0
Sector_Std             0
Source                 0
dtype: int64


In [ ]:
# ------------------------------------------------------------
# 3. Target and predictors
# ------------------------------------------------------------

df["overrun_pct"] = pd.to_numeric(
    df["Cost_Overrun_Pct"],
    errors="raise"
)

df["orig_size_m"] = pd.to_numeric(
    df["Orig_USD_PPP_2025_M"],
    errors="raise"
)

# Project size must be positive for logarithmic transformation
if (df["orig_size_m"] <= 0).any():
    raise ValueError(
        "Orig_USD_PPP_2025_M contains zero or negative values."
    )

# Log project size
df["log_size"] = np.log(df["orig_size_m"])

# Log-transformed target
#
# Cost overrun is represented as a proportion inside log1p:
# e.g. 20% -> log(1.20)
#
# This allows zero overrun observations to remain valid.
if (df["overrun_pct"] <= -100).any():
    raise ValueError(
        "Cost_Overrun_Pct contains a value <= -100%, "
        "which is invalid for log1p(1 + overrun/100)."
    )

df["y_log"] = np.log1p(
    df["overrun_pct"] / 100.0
)

NUM = [
    "log_size"
]

CAT = [
    "Sector_Std",
    "Source"
]

FEATURES = NUM + CAT

X = df[FEATURES].copy()

y_pct = df["overrun_pct"].to_numpy(dtype=float)
y_log = df["y_log"].to_numpy(dtype=float)

n = len(df)

# Project identifier
id_candidates = [
    "Project_ID",
    "Project",
    "Project_Name",
    "Name"
]

ID_COL = next(
    (c for c in id_candidates if c in df.columns),
    None
)

if ID_COL is not None:
    project_ids = (
        df[ID_COL]
        .astype(str)
        .to_numpy()
    )
else:
    project_ids = np.array(
        [f"Project_{i+1:03d}" for i in range(n)]
    )

print(f"\nPredictors: {FEATURES}")
print(f"Target: ln(1 + Cost_Overrun_Pct / 100)")
print(f"Number of projects: {n}")




Predictors: ['log_size', 'Sector_Std', 'Source']
Target: ln(1 + Cost_Overrun_Pct / 100)
Number of projects: 107


In [ ]:
# ------------------------------------------------------------
# 4. Model pipelines
# ------------------------------------------------------------

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            NUM
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            CAT
        ),
    ]
)

MODELS = [

    (
        "On-Budget Baseline",

        Pipeline([
            (
                "model",
                DummyRegressor(
                    strategy="constant",
                    constant=0.0
                )
            )
        ]),

        False
    ),

    (
        "Linear Regression",

        Pipeline([
            (
                "pre",
                preprocess
            ),
            (
                "model",
                LinearRegression()
            )
        ]),

        True
    ),

    (
        "XGBoost",

        Pipeline([
            (
                "pre",
                preprocess
            ),
            (
                "model",
                XGBRegressor(
                    n_estimators=200,
                    max_depth=2,
                    learning_rate=0.05,
                    min_child_weight=5,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.1,
                    reg_lambda=5.0,
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    objective="reg:squarederror"
                )
            )
        ]),

        True
    ),
]

In [ ]:
# ------------------------------------------------------------
# 5. Performance metrics
# ------------------------------------------------------------

def calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    error = predicted - actual

    return {

        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            )
        ),

        "MAE": float(
            mean_absolute_error(
                actual,
                predicted
            )
        ),

        "Median_AE": float(
            median_absolute_error(
                actual,
                predicted
            )
        ),

        "R2": float(
            r2_score(
                actual,
                predicted
            )
        ),

        "Bias": float(
            np.mean(error)
        ),

        "Within_10pp": float(
            np.mean(
                np.abs(error) <= 10
            )
        ),

        "Within_25pp": float(
            np.mean(
                np.abs(error) <= 25
            )
        ),
    }

In [ ]:
# ------------------------------------------------------------
# Optional: Duan smearing retransformation
# ------------------------------------------------------------

def duan_smearing_factor(
    fitted_model,
    X_train,
    y_train_log
):
    """
    Estimate the Duan smearing factor from training residuals.

    The factor is calculated only from the training fold to
    prevent information leakage.
    """

    fitted_train = fitted_model.predict(X_train)

    residuals = (
        y_train_log - fitted_train
    )

    return float(
        np.mean(
            np.exp(residuals)
        )
    )

In [ ]:
# ------------------------------------------------------------
# 6. Repeated out-of-fold validation
# ------------------------------------------------------------

per_repeat = []

# Predictions:
#
# dimensions =
#   [repeat, project]
#
# Each project receives exactly one OOF prediction
# in every repetition.

all_predictions = {
    name: np.full(
        (N_REPEATS, n),
        np.nan
    )
    for name, _, _ in MODELS
}

# Optional storage of fold-specific smearing factors
smearing_records = []

print(
    f"Primary sample size: N={n}"
)

print(
    f"Validation: "
    f"{N_SPLITS}-fold CV × "
    f"{N_REPEATS} repeats"
)

print(
    "Each project receives "
    f"{N_REPEATS} out-of-fold predictions per model."
)

for model_name, prototype, on_log_scale in MODELS:

    print(
        f"\n========== {model_name} =========="
    )

    for rep in range(N_REPEATS):

        kf = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=(
                RANDOM_STATE + rep
            )
        )

        oof = np.full(
            n,
            np.nan
        )

        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            kf.split(X),
            start=1
        ):

            model = clone(
                prototype
            )

            if on_log_scale:

                # ----------------------------------------
                # Fit on transformed target
                # ----------------------------------------

                model.fit(
                    X.iloc[train_idx],
                    y_log[train_idx]
                )

                pred_log = model.predict(
                    X.iloc[test_idx]
                )

                # ----------------------------------------
                # Back-transform
                # ----------------------------------------

                pred_ratio = np.exp(
                    pred_log
                )

                if APPLY_DUAN_SMEARING:

                    smear = (
                        duan_smearing_factor(
                            model,
                            X.iloc[train_idx],
                            y_log[train_idx]
                        )
                    )

                    pred_ratio = (
                        pred_ratio * smear
                    )

                    smearing_records.append({
                        "Model": model_name,
                        "Repeat": rep + 1,
                        "Fold": fold,
                        "Smearing_Factor": smear
                    })

                # Return to percentage points
                pred = (
                    pred_ratio - 1
                ) * 100.0

            else:

                # Baseline is already on
                # percentage-point scale

                model.fit(
                    X.iloc[train_idx],
                    y_pct[train_idx]
                )

                pred = model.predict(
                    X.iloc[test_idx]
                )

            oof[test_idx] = pred

        # ----------------------------------------
        # Verify complete OOF coverage
        # ----------------------------------------

        if np.isnan(oof).any():

            raise RuntimeError(
                f"Missing OOF predictions for "
                f"{model_name}, "
                f"repeat {rep + 1}"
            )

        # Store project-level predictions
        all_predictions[
            model_name
        ][rep, :] = oof

        # ----------------------------------------
        # Score this repetition
        # ----------------------------------------

        metrics = calculate_metrics(
            y_pct,
            oof
        )

        per_repeat.append({
            "Model": model_name,
            "Repeat": rep + 1,
            **metrics
        })

per_repeat_df = pd.DataFrame(
    per_repeat
)

print("\nRepeated CV completed.")

print(
    f"Total project-level OOF predictions per "
    f"model = {N_REPEATS * n}"
)

Primary sample size: N=107
Validation: 10-fold CV × 20 repeats
Each project receives 20 out-of-fold predictions per model.

========== On-Budget Baseline ==========

========== Linear Regression ==========

========== XGBoost ==========

Repeated CV completed.
Total project-level OOF predictions per model = 2140


In [ ]:
# ------------------------------------------------------------
# 7. Aggregate predictive performance
# ------------------------------------------------------------

summary_rows = []

for model_name, _, _ in MODELS:

    sub = per_repeat_df[
        per_repeat_df["Model"] == model_name
    ]

    summary_rows.append({

        "Model": model_name,

        "RMSE mean":
            sub["RMSE"].mean(),

        "RMSE sd":
            sub["RMSE"].std(
                ddof=1
            ),

        "MAE mean":
            sub["MAE"].mean(),

        "MAE sd":
            sub["MAE"].std(
                ddof=1
            ),

        "Median AE mean":
            sub["Median_AE"].mean(),

        "Median AE sd":
            sub["Median_AE"].std(
                ddof=1
            ),

        "R2 mean":
            sub["R2"].mean(),

        "R2 sd":
            sub["R2"].std(
                ddof=1
            ),

        "Bias mean":
            sub["Bias"].mean(),

        "Bias sd":
            sub["Bias"].std(
                ddof=1
            ),

        "Within ±10pp mean":
            sub["Within_10pp"].mean(),

        "Within ±25pp mean":
            sub["Within_25pp"].mean(),

        "Repeats":
            len(sub)
    })

summary_df = pd.DataFrame(
    summary_rows
)

print(
    summary_df.round(3).to_string(
        index=False
    )
)

             Model  RMSE mean  RMSE sd  MAE mean  MAE sd  Median AE mean  Median AE sd  R2 mean  R2 sd  Bias mean  Bias sd  Within ±10pp mean  Within ±25pp mean  Repeats
On-Budget Baseline     64.664    0.000    26.391   0.000           9.470         0.000   -0.127  0.000    -21.687    0.000              0.523              0.720       20
 Linear Regression     58.239    0.508    27.044   0.328          17.604         0.612    0.086  0.016     -6.036    0.286              0.339              0.714       20
           XGBoost     59.273    0.678    26.838   0.523          14.562         0.770    0.053  0.022     -5.571    0.484              0.364              0.761       20


In [ ]:
# ------------------------------------------------------------
# 8. Project-level paired errors
# ------------------------------------------------------------

project_predictions = pd.DataFrame({
    "Project_ID": project_ids,
    "Actual_Overrun_pct": y_pct
})

for model_name, _, _ in MODELS:

    arr = all_predictions[
        model_name
    ]

    # Mean prediction across the 20 repetitions
    project_predictions[
        f"{model_name}_MeanPrediction"
    ] = arr.mean(axis=0)

    # Mean absolute error for each project
    # across the 20 repeated OOF predictions
    project_predictions[
        f"{model_name}_MeanAbsError"
    ] = np.mean(
        np.abs(
            arr -
            y_pct.reshape(1, -1)
        ),
        axis=0
    )

# ------------------------------------------------------------
# LR vs XGBoost
# ------------------------------------------------------------

lr_mae = (
    project_predictions[
        "Linear Regression_MeanAbsError"
    ]
)

xgb_mae = (
    project_predictions[
        "XGBoost_MeanAbsError"
    ]
)

# Paired project-level difference
#
# Positive value:
# LR has greater error than XGBoost
#
# Negative value:
# LR has lower error than XGBoost

paired_difference = (
    lr_mae - xgb_mae
)

print(
    "\nProject-level paired comparison:"
)

print(
    f"Mean LR MAE: "
    f"{lr_mae.mean():.3f}"
)

print(
    f"Mean XGBoost MAE: "
    f"{xgb_mae.mean():.3f}"
)

print(
    f"Mean difference (LR - XGB): "
    f"{paired_difference.mean():.3f}"
)

print(
    f"Median difference: "
    f"{np.median(paired_difference):.3f}"
)


Project-level paired comparison:
Mean LR MAE: 27.044
Mean XGBoost MAE: 26.838
Mean difference (LR - XGB): 0.207
Median difference: 1.004


In [ ]:
# ------------------------------------------------------------
# 9. Paired bootstrap confidence interval
# ------------------------------------------------------------

def paired_bootstrap_mean_difference(
    differences,
    n_resamples=20000,
    random_state=42
):

    differences = np.asarray(
        differences,
        dtype=float
    )

    rng = np.random.default_rng(
        random_state
    )

    n_obs = len(differences)

    # Resample projects, not repeated CV predictions
    indices = rng.integers(
        low=0,
        high=n_obs,
        size=(n_resamples, n_obs)
    )

    bootstrap_means = (
        differences[indices].mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    return {
        "estimate":
            float(differences.mean()),

        "ci_low":
            float(ci_low),

        "ci_high":
            float(ci_high)
    }


bootstrap_result = (
    paired_bootstrap_mean_difference(
        paired_difference,
        n_resamples=N_BOOTSTRAP,
        random_state=RANDOM_STATE
    )
)

print("\n========== PAIRED BOOTSTRAP ==========")

print(
    f"Mean MAE difference (LR - XGB): "
    f"{bootstrap_result['estimate']:.3f} pp"
)

print(
    f"95% bootstrap CI: "
    f"[{bootstrap_result['ci_low']:.3f}, "
    f"{bootstrap_result['ci_high']:.3f}] pp"
)


========== PAIRED BOOTSTRAP ==========
Mean MAE difference (LR - XGB): 0.207 pp
95% bootstrap CI: [-1.478, 1.848] pp


In [ ]:
# ------------------------------------------------------------
# 10. Paired permutation test
# ------------------------------------------------------------

def paired_permutation_test(
    differences,
    n_resamples=20000,
    random_state=42
):

    differences = np.asarray(
        differences,
        dtype=float
    )

    observed = differences.mean()

    rng = np.random.default_rng(
        random_state
    )

    # Randomly reverse the sign of each
    # project-level paired difference.
    #
    # This preserves the pairing between
    # Linear Regression and XGBoost.

    signs = rng.choice(
        [-1.0, 1.0],
        size=(
            n_resamples,
            len(differences)
        )
    )

    permuted = (
        (signs * differences)
        .mean(axis=1)
    )

    # Two-sided p-value
    p_value = (
        np.sum(
            np.abs(permuted)
            >= abs(observed)
        ) + 1
    ) / (
        n_resamples + 1
    )

    return {
        "observed_difference":
            float(observed),

        "p_value":
            float(p_value)
    }


permutation_result = (
    paired_permutation_test(
        paired_difference,
        n_resamples=N_PERMUTATIONS,
        random_state=RANDOM_STATE
    )
)

print("\n========== PAIRED PERMUTATION TEST ==========")

print(
    f"Observed mean difference: "
    f"{permutation_result['observed_difference']:.3f} pp"
)

print(
    f"Two-sided permutation p-value: "
    f"{permutation_result['p_value']:.4f}"
)


========== PAIRED PERMUTATION TEST ==========
Observed mean difference: 0.207 pp
Two-sided permutation p-value: 0.8091


In [ ]:
# ------------------------------------------------------------
# 11. Wilcoxon signed-rank robustness test
# ------------------------------------------------------------

try:

    w_stat, w_p = wilcoxon(
        lr_mae,
        xgb_mae,
        alternative="two-sided",
        zero_method="wilcox"
    )

except ValueError:

    w_stat = np.nan
    w_p = np.nan

print(
    "\n========== WILCOXON ROBUSTNESS TEST =========="
)

print(
    f"Wilcoxon statistic: "
    f"{w_stat:.3f}"
)

print(
    f"Wilcoxon p-value: "
    f"{w_p:.4f}"
)


========== WILCOXON ROBUSTNESS TEST ==========
Wilcoxon statistic: 2625.000
Wilcoxon p-value: 0.4119


In [ ]:
# ------------------------------------------------------------
# 12. Model comparison table
# ------------------------------------------------------------

paired_comparison = pd.DataFrame({

    "Statistic": [

        "Number of projects",

        "Mean project-level MAE: Linear Regression",

        "Mean project-level MAE: XGBoost",

        "Mean MAE difference (LR - XGBoost)",

        "95% bootstrap CI lower",

        "95% bootstrap CI upper",

        "Paired permutation p-value",

        "Wilcoxon statistic",

        "Wilcoxon p-value",
    ],

    "Value": [

        n,

        float(
            lr_mae.mean()
        ),

        float(
            xgb_mae.mean()
        ),

        bootstrap_result[
            "estimate"
        ],

        bootstrap_result[
            "ci_low"
        ],

        bootstrap_result[
            "ci_high"
        ],

        permutation_result[
            "p_value"
        ],

        float(w_stat)
        if not np.isnan(w_stat)
        else np.nan,

        float(w_p)
        if not np.isnan(w_p)
        else np.nan,
    ]
})

print(
    paired_comparison.to_string(
        index=False
    )
)

                                Statistic       Value
                       Number of projects  107.000000
Mean project-level MAE: Linear Regression   27.044267
          Mean project-level MAE: XGBoost   26.837553
       Mean MAE difference (LR - XGBoost)    0.206715
                   95% bootstrap CI lower   -1.478198
                   95% bootstrap CI upper    1.848228
               Paired permutation p-value    0.809060
                       Wilcoxon statistic 2625.000000
                         Wilcoxon p-value    0.411923


In [ ]:
# ------------------------------------------------------------
# 13. Headline RQ2 results
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("RQ2 HEADLINE RESULTS")
print("=" * 70)

for model in [
    "Linear Regression",
    "XGBoost"
]:

    row = summary_df[
        summary_df["Model"] == model
    ].iloc[0]

    print(f"\n{model}")

    print(
        f"  RMSE: "
        f"{row['RMSE mean']:.2f} "
        f"(SD {row['RMSE sd']:.2f})"
    )

    print(
        f"  MAE: "
        f"{row['MAE mean']:.2f} "
        f"(SD {row['MAE sd']:.2f})"
    )

    print(
        f"  Median AE: "
        f"{row['Median AE mean']:.2f}"
    )

    print(
        f"  R²: "
        f"{row['R2 mean']:.3f} "
        f"(SD {row['R2 sd']:.3f})"
    )

    print(
        f"  Bias: "
        f"{row['Bias mean']:.2f} pp"
    )

    print(
        f"  Within ±10pp: "
        f"{row['Within ±10pp mean']:.1%}"
    )

    print(
        f"  Within ±25pp: "
        f"{row['Within ±25pp mean']:.1%}"
    )

print("\n")
print("LR - XGB mean MAE difference: "
      f"{bootstrap_result['estimate']:.3f} pp")

print(
    "95% bootstrap CI: "
    f"[{bootstrap_result['ci_low']:.3f}, "
    f"{bootstrap_result['ci_high']:.3f}] pp"
)

print(
    "Paired permutation p-value: "
    f"{permutation_result['p_value']:.4f}"
)

print(
    "Wilcoxon p-value: "
    f"{w_p:.4f}"
)



RQ2 HEADLINE RESULTS

Linear Regression
  RMSE: 58.24 (SD 0.51)
  MAE: 27.04 (SD 0.33)
  Median AE: 17.60
  R²: 0.086 (SD 0.016)
  Bias: -6.04 pp
  Within ±10pp: 33.9%
  Within ±25pp: 71.4%

XGBoost
  RMSE: 59.27 (SD 0.68)
  MAE: 26.84 (SD 0.52)
  Median AE: 14.56
  R²: 0.053 (SD 0.022)
  Bias: -5.57 pp
  Within ±10pp: 36.4%
  Within ±25pp: 76.1%


LR - XGB mean MAE difference: 0.207 pp
95% bootstrap CI: [-1.478, 1.848] pp
Paired permutation p-value: 0.8091
Wilcoxon p-value: 0.4119


In [ ]:
# ------------------------------------------------------------
# 14. Model-selection summary
# ------------------------------------------------------------

lr_row = summary_df[
    summary_df["Model"] == "Linear Regression"
].iloc[0]

xgb_row = summary_df[
    summary_df["Model"] == "XGBoost"
].iloc[0]

selection_summary = pd.DataFrame({

    "Metric": [
        "RMSE",
        "MAE",
        "Median AE",
        "R²",
        "Absolute Bias",
        "Within ±10pp",
        "Within ±25pp"
    ],

    "Linear Regression": [
        lr_row["RMSE mean"],
        lr_row["MAE mean"],
        lr_row["Median AE mean"],
        lr_row["R2 mean"],
        abs(lr_row["Bias mean"]),
        lr_row["Within ±10pp mean"],
        lr_row["Within ±25pp mean"]
    ],

    "XGBoost": [
        xgb_row["RMSE mean"],
        xgb_row["MAE mean"],
        xgb_row["Median AE mean"],
        xgb_row["R2 mean"],
        abs(xgb_row["Bias mean"]),
        xgb_row["Within ±10pp mean"],
        xgb_row["Within ±25pp mean"]
    ]
})

print(
    "\n========== MODEL SELECTION =========="
)

print(
    selection_summary.round(3).to_string(
        index=False
    )
)


========== MODEL SELECTION ==========
       Metric  Linear Regression  XGBoost
         RMSE             58.239   59.273
          MAE             27.044   26.838
    Median AE             17.604   14.562
           R²              0.086    0.053
Absolute Bias              6.036    5.571
 Within ±10pp              0.339    0.364
 Within ±25pp              0.714    0.761


In [ ]:
# ------------------------------------------------------------
# 15. Dissertation-ready figures
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

BASELINE = "On-Budget Baseline"
MODEL_BARS = ["Linear Regression", "XGBoost"]
ALL_ORDER = [BASELINE, "Linear Regression", "XGBoost"]
hl = summary_df.set_index("Model")

def save_fig(fig, filename):
    fig.savefig(os.path.join(FIG_DIR, filename + ".png"))
    fig.savefig(os.path.join(FIG_DIR, filename + ".svg"))
    plt.close(fig)

# Figure 1: headline — two models as bars, on-budget baseline as dotted line
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

mae = hl.loc[MODEL_BARS, "MAE mean"].to_numpy()
bmae = hl.loc[BASELINE, "MAE mean"]
axes[0].bar(MODEL_BARS, mae)
axes[0].axhline(bmae, linestyle="--", color="black", linewidth=1.5,
                label=f"On-budget baseline ({bmae:.1f})")
axes[0].set_ylabel("Mean absolute error (percentage points)")
axes[0].set_title("Average prediction error\nLower is better")
axes[0].tick_params(axis="x", rotation=15); axes[0].legend()
for i, v in enumerate(mae): axes[0].text(i, v + 0.2, f"{v:.1f}", ha="center")

w10 = hl.loc[MODEL_BARS, "Within ±10pp mean"].to_numpy()
b10 = hl.loc[BASELINE, "Within ±10pp mean"]
axes[1].bar(MODEL_BARS, w10)
axes[1].axhline(b10, linestyle="--", color="black", linewidth=1.5,
                label=f"On-budget baseline ({b10:.0%})")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Share of projects within ±10pp")
axes[1].set_title("Predictions within ±10 percentage points\nHigher is better")
axes[1].tick_params(axis="x", rotation=15); axes[1].legend()
for i, v in enumerate(w10): axes[1].text(i, v + 0.01, f"{v:.0%}", ha="center")

fig.suptitle("Linear Regression and XGBoost against the on-budget baseline", fontweight="bold")
fig.tight_layout(); save_fig(fig, "Fig1_headline_performance")

# Figure 2: predicted vs actual (mean prediction across repeats)
# Mean OOF prediction across repeated CV runs
avg_preds = {
    name: all_predictions[name].mean(axis=0)
    for name, _, _ in MODELS
}
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, name in zip(axes, ["Linear Regression", "XGBoost"]):
    pred = avg_preds[name]
    lo = min(y_pct.min(), pred.min()); hi = max(y_pct.max(), pred.max())
    ax.scatter(y_pct, pred, alpha=0.65)
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5)
    ax.set_title(name); ax.set_xlabel("Actual cost overrun (%)")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
axes[0].set_ylabel("Predicted cost overrun (%)")
fig.suptitle("Mean out-of-fold prediction versus actual cost overrun", fontweight="bold")
fig.tight_layout(); save_fig(fig, "Fig2_predicted_vs_actual")

# Figure 3: practical accuracy — Linear Regression vs XGBoost head-to-head (baseline removed)
fig, ax = plt.subplots(figsize=(8.5, 5))
x = np.arange(len(MODEL_BARS)); width = 0.36
b1 = ax.bar(x - width/2, hl.loc[MODEL_BARS, "Within ±10pp mean"], width, label="Within ±10 percentage points")
b2 = ax.bar(x + width/2, hl.loc[MODEL_BARS, "Within ±25pp mean"], width, label="Within ±25 percentage points")
ax.set_xticks(x); ax.set_xticklabels(MODEL_BARS, rotation=0)
ax.set_ylim(0, 1); ax.set_ylabel("Share of projects")
ax.set_title("How often were predictions reasonably close to the actual overrun?", fontweight="bold")
ax.legend()
for container in [b1, b2]:
    ax.bar_label(container, labels=[f"{v:.0%}" for v in container.datavalues], padding=3)
fig.tight_layout(); save_fig(fig, "Fig3_practical_accuracy")

# Figure 4: prediction error distribution
fig, ax = plt.subplots(figsize=(8.5, 5))
error_data = [avg_preds[name] - y_pct for name in ["Linear Regression", "XGBoost"]]
ax.boxplot(error_data, tick_labels=["Linear Regression", "XGBoost"], showmeans=True)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_ylabel("Prediction error (Predicted - Actual), percentage points")
ax.set_title("Distribution of prediction errors", fontweight="bold")
fig.tight_layout(); save_fig(fig, "Fig4_prediction_error")

# Figure 5: R2 stability across repeats
fig, ax = plt.subplots(figsize=(9, 5))
for name in ["Linear Regression", "XGBoost"]:
    sub = per_repeat_df[per_repeat_df["Model"] == name]
    ax.plot(sub["Repeat"], sub["R2"], marker="o", label=name)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Repeated cross-validation iteration"); ax.set_ylabel("R²")
ax.set_title("Stability of predictive performance across repeated cross-validation", fontweight="bold")
ax.legend(); fig.tight_layout(); save_fig(fig, "Fig5_r2_stability")

# Figure 6: target distribution
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.hist(y_pct, bins=20, edgecolor="black")
ax.axvline(np.mean(y_pct), linestyle="--", linewidth=1.5, label=f"Mean = {np.mean(y_pct):.1f}%")
ax.axvline(np.median(y_pct), linestyle=":", linewidth=1.5, label=f"Median = {np.median(y_pct):.1f}%")
ax.set_xlabel("Actual cost overrun (%)"); ax.set_ylabel("Number of projects")
ax.set_title("Distribution of cost overruns in the study sample", fontweight="bold")
ax.legend(); fig.tight_layout(); save_fig(fig, "Fig6_overrun_distribution")

# Figure 7: bias comparison (models only; a constant-0 baseline has trivial bias = -mean)
fig, ax = plt.subplots(figsize=(8, 5))
bias = hl.loc[MODEL_BARS, "Bias mean"].to_numpy()
ax.bar(MODEL_BARS, bias)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_ylabel("Mean prediction bias (percentage points)")
ax.set_title("Systematic overprediction or underprediction", fontweight="bold")
ax.tick_params(axis="x", rotation=15)
for i, v in enumerate(bias): ax.text(i, v + (0.1 if v >= 0 else -0.3), f"{v:.1f}", ha="center")
fig.tight_layout(); save_fig(fig, "Fig7_bias")

# ------------------------------------------------------------
# Figure 8: paired project-level MAE difference
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 5)
)

ax.hist(
    paired_difference,
    bins=20,
    edgecolor="black"
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.2,
    label="No difference"
)

ax.axvline(
    paired_difference.mean(),
    linestyle="-",
    linewidth=1.5,
    label=(
        f"Mean difference = "
        f"{paired_difference.mean():.2f} pp"
    )
)

ax.set_xlabel(
    "Project-level MAE difference "
    "(Linear Regression − XGBoost), pp"
)

ax.set_ylabel(
    "Number of projects"
)

ax.set_title(
    "Paired project-level prediction-error difference",
    fontweight="bold"
)

ax.legend()

fig.tight_layout()

save_fig(
    fig,
    "Fig8_paired_MAE_difference"
)

print(f"\nSaved figures to: {FIG_DIR}/")
print("Completed: On-Budget Baseline + Linear Regression + XGBoost.")


Saved figures to: RQ2_Figures_Corrected/
Completed: On-Budget Baseline + Linear Regression + XGBoost.


In [ ]:
# ------------------------------------------------------------
# 16. Excel workbook  (replaces the previous cells 17 and 18)
# Writes: RQ2_Summary, RQ2_PerRepeat, RQ2_ProjectPredictions,
#         RQ2_StatisticalComparison, Config  — then saves.
# ------------------------------------------------------------

from openpyxl.utils import get_column_letter

BOLD   = Font(name="Arial", size=10, bold=True)
ARIAL  = Font(name="Arial", size=10)
HEADER = PatternFill("solid", fgColor="D9D9D9")
CENTER = Alignment(horizontal="center", vertical="center")
THIN   = Side(style="thin", color="BFBFBF")
BOX    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def style_header(ws, ncols):
    for c in range(1, ncols + 1):
        cell = ws.cell(1, c)
        cell.font = BOLD; cell.fill = HEADER
        cell.alignment = CENTER; cell.border = BOX

def style_sheet(ws):
    for row in ws.iter_rows():
        for cell in row:
            cell.font = ARIAL; cell.border = BOX
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb = openpyxl.Workbook()

# ---- RQ2_Summary ----
ws = wb.active; ws.title = "RQ2_Summary"
summary_cols = list(summary_df.columns); ws.append(summary_cols)
for _, row in summary_df.iterrows(): ws.append(row.tolist())
style_header(ws, len(summary_cols)); style_sheet(ws)
for c in range(2, len(summary_cols) + 1):
    header = ws.cell(1, c).value
    for r in range(2, ws.max_row + 1):
        if "Within" in header: ws.cell(r, c).number_format = "0.0%"
        elif "R2" in header:   ws.cell(r, c).number_format = "0.000"
        elif header == "Repeats": ws.cell(r, c).number_format = "0"
        else: ws.cell(r, c).number_format = "0.00"
for col, width in {"A":22,"B":13,"C":11,"D":13,"E":11,"F":16,"G":14,
                   "H":11,"I":10,"J":13,"K":12,"L":18,"M":18,"N":9}.items():
    ws.column_dimensions[col].width = width

# ---- RQ2_PerRepeat ----
ws = wb.create_sheet("RQ2_PerRepeat")
per_cols = ["Model","Repeat","RMSE","MAE","Median_AE","R2","Bias","Within_10pp","Within_25pp"]
ws.append(per_cols)
for _, row in per_repeat_df.iterrows(): ws.append([row[c] for c in per_cols])
style_header(ws, len(per_cols)); style_sheet(ws)
for r in range(2, ws.max_row + 1):
    for c in [3,4,5,7]: ws.cell(r,c).number_format = "0.00"
    ws.cell(r,6).number_format = "0.000"
    for c in [8,9]: ws.cell(r,c).number_format = "0.0%"
for col in range(1,10): ws.column_dimensions[get_column_letter(col)].width = 16

# ---- RQ2_ProjectPredictions ----
ws = wb.create_sheet("RQ2_ProjectPredictions")
for c in project_predictions.columns:
    ws.cell(1, project_predictions.columns.get_loc(c)+1, c)
for r_idx, (_, row) in enumerate(project_predictions.iterrows(), start=2):
    for c_idx, value in enumerate(row, start=1): ws.cell(r_idx, c_idx, value)
style_header(ws, len(project_predictions.columns)); style_sheet(ws)
for c in range(2, ws.max_column + 1):
    for r in range(2, ws.max_row + 1): ws.cell(r,c).number_format = "0.00"
for c in range(1, ws.max_column + 1):
    ws.column_dimensions[get_column_letter(c)].width = 25

# ---- RQ2_StatisticalComparison ----
ws = wb.create_sheet("RQ2_StatisticalComparison")
ws.append(list(paired_comparison.columns))
for _, row in paired_comparison.iterrows(): ws.append(row.tolist())
style_header(ws, 2); style_sheet(ws)
ws.column_dimensions["A"].width = 45; ws.column_dimensions["B"].width = 20
for r in range(2, ws.max_row + 1): ws.cell(r,2).number_format = "0.0000"

# ---- Config ----
ws = wb.create_sheet("Config")
config_rows = [
    ("Analysis", "RQ2 – Out-of-sample predictive comparison"),
    ("Research Question",
     "Under rigorous out-of-sample validation, how does XGBoost compare with "
     "Linear Regression in predicting megaproject cost overruns, and what does "
     "the comparison imply for model selection under small, right-skewed samples?"),
    ("Analytical sample", f"{n} megaprojects (Meets_500M_2025 = Yes)"),
    ("Predictors", "log(Orig_USD_PPP_2025_M), Sector_Std, Source"),
    ("Country", "Excluded: nests inside Source, so collinear by construction"),
    ("Validation framework", f"{N_SPLITS}-fold cross-validation repeated {N_REPEATS} times"),
    ("Validation scoring",
     f"Each repetition produces {n} project-level out-of-fold predictions per model; "
     "metrics are computed per repetition and averaged across repetitions."),
    ("Total OOF predictions per model", f"{n * N_REPEATS}"),
    ("Inferential unit", f"Project (N={n}); repeated CV predictions are not treated as independent."),
    ("Primary model comparison", "Linear Regression versus XGBoost"),
    ("Benchmark",
     "On-Budget Baseline (0% overrun), retained as a descriptive benchmark and "
     "excluded from the LR-versus-XGBoost inferential comparison."),
    ("Target transformation", "ln(1 + Cost_Overrun_Pct / 100)"),
    ("Prediction scale", "Predictions back-transformed to percentage points before scoring."),
    ("Primary performance metrics", "RMSE and MAE"),
    ("Secondary metrics", "Median AE, R², Bias, within ±10pp, within ±25pp"),
    ("Primary paired inference",
     "Project-level paired permutation test on the mean MAE difference (LR vs XGBoost)."),
    ("Effect estimate", "Mean project-level MAE difference (LR minus XGBoost), percentage points."),
    ("Uncertainty estimate", "95% paired bootstrap confidence interval over projects."),
    ("Robustness test", "Wilcoxon signed-rank test on project-level mean absolute errors."),
    ("Bootstrap resamples", f"{N_BOOTSTRAP:,}"),
    ("Permutation resamples", f"{N_PERMUTATIONS:,}"),
    ("Duan smearing", f"APPLY_DUAN_SMEARING = {APPLY_DUAN_SMEARING} (off = primary; on = sensitivity)."),
    ("XGBoost parameters",
     "n_estimators=200; max_depth=2; learning_rate=0.05; min_child_weight=5; "
     "subsample=0.8; colsample_bytree=0.8; reg_alpha=0.1; reg_lambda=5.0"),
    ("R2 note",
     "R² is defined relative to the sample mean; the On-Budget baseline predicts 0, "
     "so its R² is negative by construction and is not a benchmark for R²."),
    ("Random seed", str(RANDOM_STATE)),
]
ws.append(["Parameter", "Value"])
for parameter, value in config_rows: ws.append([parameter, value])
style_header(ws, 2); style_sheet(ws)
ws.column_dimensions["A"].width = 32; ws.column_dimensions["B"].width = 110
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    row[0].font = BOLD
    row[0].alignment = Alignment(vertical="top", wrap_text=True)
    row[1].alignment = Alignment(vertical="top", wrap_text=True)
ws.freeze_panes = "A2"

wb.save(OUTPUT_XLSX)
print(f"Saved results workbook: {OUTPUT_XLSX}")
print("Sheets:", wb.sheetnames)

Saved results workbook: RQ2_Model_Results_Corrected.xlsx
Sheets: ['RQ2_Summary', 'RQ2_PerRepeat', 'RQ2_ProjectPredictions', 'RQ2_StatisticalComparison', 'Config']
